In [ ]:
import cv2
import numpy as np
import os
import time
from datetime import datetime
from ultralytics import YOLO

# -------- CONFIG --------
SOURCE = "C:/Users/soura/Downloads/sb/Channel ID_18_3.mp4"  # or RTSP

MIN_AREA = 1500
FRAME_SKIP = 2
YOLO_CONF = 0.4

# MIN_AREA = 500
# YOLO_CONF = 0.25
# FRAME_SKIP = 1

REDETECT_INTERVAL = 10
COOLDOWN_SEC = 2

SAVE_DIR = "dumps"
os.makedirs(SAVE_DIR, exist_ok=True)

# -------- INIT --------
cap = cv2.VideoCapture(SOURCE)
model = YOLO("yolov8n.pt")

ret, prev_frame = cap.read()
if not ret:
    raise Exception("Cannot read source")

prev_gray = cv2.cvtColor(prev_frame, cv2.COLOR_BGR2GRAY)
prev_gray = cv2.GaussianBlur(prev_gray, (21, 21), 0)

frame_id = 0
tracker = None
tracking = False
last_detect_frame = 0
last_event_time = 0

# -------- LOOP --------
while True:
    ret, frame = cap.read()
    if not ret:
        break

    frame_id += 1
    if frame_id % FRAME_SKIP != 0:
        continue

    # ---- MOTION DETECTION ----
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    gray = cv2.GaussianBlur(gray, (21, 21), 0)

    delta = cv2.absdiff(prev_gray, gray)
    thresh = cv2.threshold(delta, 25, 255, cv2.THRESH_BINARY)[1]
    thresh = cv2.dilate(thresh, None, iterations=2)

    cv2.imshow("thresh", thresh)

    contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    motion_boxes = []
    for cnt in contours:
        if cv2.contourArea(cnt) < MIN_AREA:
            continue
        x, y, w, h = cv2.boundingRect(cnt)
        motion_boxes.append((x, y, w, h))

    # ---- AGGREGATE ----
    aggregated_box = None
    if motion_boxes:
        x_min = min([x for x, y, w, h in motion_boxes])
        y_min = min([y for x, y, w, h in motion_boxes])
        x_max = max([x+w for x, y, w, h in motion_boxes])
        y_max = max([y+h for x, y, w, h in motion_boxes])

        aggregated_box = (x_min, y_min, x_max - x_min, y_max - y_min)
        cv2.rectangle(frame, (x_min, y_min), (x_max, y_max), (255, 0, 0), 2)

    # =========================================================
    # TRACKING MODE
    # =========================================================
    if tracking and tracker is not None:
        ok, bbox = tracker.update(frame)

        if ok:
            x, y, w, h = map(int, bbox)
            cv2.rectangle(frame, (x, y), (x+w, y+h), (0, 255, 0), 2)

            if frame_id - last_detect_frame > REDETECT_INTERVAL:
                tracking = False
        else:
            tracking = False

    # =========================================================
    # DETECTION MODE
    # =========================================================
    if not tracking and aggregated_box:
        x, y, w, h = aggregated_box

        h_frame, w_frame = frame.shape[:2]
        x = max(0, x)
        y = max(0, y)
        w = min(w, w_frame - x)
        h = min(h, h_frame - y)

        roi = frame[y:y+h, x:x+w]

        if roi.size > 0:
            # results = model(roi, classes=[0], conf=YOLO_CONF)
            results = model(roi, conf=YOLO_CONF, verbose=False)

            for r in results:
                boxes = r.boxes.xyxy.cpu().numpy()
                scores = r.boxes.conf.cpu().numpy()

                if len(boxes) == 0:
                    continue

                i = np.argmax(scores)

                x1, y1, x2, y2 = boxes[i]
                x1_full = int(x + x1)
                y1_full = int(y + y1)
                w_box = int(x2 - x1)
                h_box = int(y2 - y1)

                # -------- EVENT LOG + SAVE --------
                now = time.time()
                if now - last_event_time > COOLDOWN_SEC:
                    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

                    print(f"[EVENT] Person detected | frame={frame_id} | time={timestamp}")

                    # save full frame
                    cv2.imwrite(f"{SAVE_DIR}/frame_{timestamp}.jpg", frame)

                    # save cropped person
                    crop = frame[y1_full:y1_full+h_box, x1_full:x1_full+w_box]
                    if crop.size > 0:
                        cv2.imwrite(f"{SAVE_DIR}/person_{timestamp}.jpg", crop)

                    last_event_time = now

                # -------- INIT TRACKER --------
                tracker = cv2.TrackerKCF_create()
                tracker.init(frame, (x1_full, y1_full, w_box, h_box))

                tracking = True
                last_detect_frame = frame_id

                break

    prev_gray = gray

    cv2.imshow("frame", frame)
    if cv2.waitKey(1) & 0xFF == 27:
        break

cap.release()
cv2.destroyAllWindows()